# Stage 1 Strict Replication Notebook

This notebook implements the paper/repo SVM baseline methodology only.

## Paper Mapping
This notebook maps directly to the baseline method in the dataset paper:
- official dataset + predefined folds
- pyAudioAnalysis feature extraction (`1s` mid-term, `50ms` short-term)
- RBF-SVM with `C/gamma` grid search
- file-level fold evaluation with confusion matrices and summary metrics


In [ ]:
from pathlib import Path
import json
import importlib
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report


### Implemented Paper Part: Dataset + Official Split Protocol
Defines the dataset root and selects the official split JSON (`folds/guitars/amplifiers`).

In [ ]:
PROJECT_ROOT = Path.cwd()

# Fixed dataset path (confirmed working)
DATA_ROOT = PROJECT_ROOT / 'guitar_style_dataset' / 'magcil-guitar_style_dataset-eb27d7b' / 'data'
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Expected DATA_ROOT not found: {DATA_ROOT}')

WAV_ROOT = DATA_ROOT / 'wav'
SPLIT_FILES = {
    'folds': DATA_ROOT / 'folds.json',
    'guitars': DATA_ROOT / 'guitars.json',
    'amplifiers': DATA_ROOT / 'amplifiers.json',
}

STRICT_STAGE1_ROOT = PROJECT_ROOT / 'outputs' / 'stage1_strict_replication'
STRICT_FEATURE_ROOT = STRICT_STAGE1_ROOT / 'features'
STRICT_RESULTS_ROOT = STRICT_STAGE1_ROOT / 'results'
STRICT_FIG_ROOT = STRICT_STAGE1_ROOT / 'figures'
for p in [STRICT_STAGE1_ROOT, STRICT_FEATURE_ROOT, STRICT_RESULTS_ROOT, STRICT_FIG_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

EVAL_SPLIT_SCHEMA = 'folds'  # 'folds' | 'guitars' | 'amplifiers'
SPLIT_JSON_PATH = SPLIT_FILES[EVAL_SPLIT_SCHEMA]

PARAM_GRID = {
    'C': [0.1, 1, 10, 50, 100, 1000],
    'gamma': [0.0001, 0.001, 0.01, 0.1, 1, 10],
    'kernel': ['rbf'],
}
INNER_CV_FOLDS = 5
GRID_N_JOBS = max(1, min(8, (os.cpu_count() or 2)))
PARALLEL_BACKEND = 'threading'
RANDOM_STATE = 42

CLASS_MAPPING = {
    'alternate picking': 0,
    'legato': 1,
    'tapping': 2,
    'sweep picking': 3,
    'vibrato': 4,
    'hammer on': 5,
    'pull off': 6,
    'slide': 7,
    'bend': 8,
}

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT   :', DATA_ROOT)
print('WAV_ROOT    :', WAV_ROOT)
print('Eval split  :', EVAL_SPLIT_SCHEMA, SPLIT_JSON_PATH)
print('Grid size   :', len(PARAM_GRID['C']) * len(PARAM_GRID['gamma']))
print('Parallel    :', PARALLEL_BACKEND, f'n_jobs={GRID_N_JOBS}')


### Implemented Paper Part: SVM Feature Construction
Extracts full-file handcrafted statistical features using pyAudioAnalysis with the baseline window settings.

In [ ]:
from pyAudioAnalysis import MidTermFeatures as aF

class_dirs = [str(d) for d in sorted(WAV_ROOT.iterdir()) if d.is_dir()]
features, class_names, file_names = aF.multiple_directory_feature_extraction(
    class_dirs,
    1.0,   # mid_window
    1.0,   # mid_step
    0.05,  # short_window
    0.05   # short_step
)

flat_file_names = [Path(x).name for sub in file_names for x in sub]
shapes = [arr.shape[0] for arr in features]
X = np.concatenate(features, axis=0).astype(np.float32)

labels = []
ordered_classes = [Path(c).name for c in class_dirs]
for cls_name, count in zip(ordered_classes, shapes):
    labels.extend([CLASS_MAPPING[cls_name]] * count)

meta_df = pd.DataFrame({'file_name': flat_file_names, 'label': labels})
meta_df['class_name'] = meta_df['label'].map({v: k for k, v in CLASS_MAPPING.items()})
meta_df = meta_df.reset_index().rename(columns={'index': 'feat_idx'})

np.save(STRICT_FEATURE_ROOT / 'X_file_features.npy', X)
meta_df.to_csv(STRICT_FEATURE_ROOT / 'file_feature_index.csv', index=False)
(STRICT_FEATURE_ROOT / 'class_mapping.json').write_text(json.dumps(CLASS_MAPPING, indent=2))

print('X shape:', X.shape)
if X.shape[1] != 136:
    print(f'WARNING: expected ~136 dims, got {X.shape[1]}')
print('Saved:', STRICT_FEATURE_ROOT / 'X_file_features.npy')
print('Saved:', STRICT_FEATURE_ROOT / 'file_feature_index.csv')


### Implemented Paper Part: Fold Assignment
Builds file-level train/test membership from the official split JSON.

In [ ]:
# -------- Build file-level fold table --------
split_obj = json.loads(SPLIT_JSON_PATH.read_text())
rows = []
for fold_name, data in split_obj.items():
    for part in ['train', 'test']:
        for f in data.get(part, []):
            rows.append({
                'fold': str(fold_name),
                'partition': part,
                'file_name': Path(str(f)).name,
            })

split_df = pd.DataFrame(rows).drop_duplicates()
meta_df = pd.read_csv(STRICT_FEATURE_ROOT / 'file_feature_index.csv')
model_df = split_df.merge(meta_df, on='file_name', how='inner')

print('Split rows:', len(split_df))
print('Matched rows:', len(model_df))
print('Unique split files:', split_df['file_name'].nunique())
print('Unique matched files:', model_df['file_name'].nunique())
missing = set(split_df['file_name']) - set(model_df['file_name'])
print('Missing files after join:', len(missing))


### Implemented Paper Part: RBF-SVM Baseline Evaluation
Runs fold-wise grid-searched SVM and reports file-level metrics and confusion matrices.

In [ ]:
# -------- Train + evaluate strict file-level SVM --------
from joblib import parallel_backend

X = np.load(STRICT_FEATURE_ROOT / 'X_file_features.npy')
model_df = model_df.copy()

folds = sorted(model_df['fold'].unique().tolist())
label_order = sorted(CLASS_MAPPING.values())
idx_to_class = {v: k for k, v in CLASS_MAPPING.items()}
class_names_ordered = [idx_to_class[i] for i in label_order]

all_fold_metrics = []
all_pred_rows = []
agg_cm = np.zeros((len(label_order), len(label_order)), dtype=int)

for fold in folds:
    fd = model_df[model_df['fold'] == fold].copy()
    tr = fd[fd['partition'] == 'train'].copy()
    te = fd[fd['partition'] == 'test'].copy()

    X_train = X[tr['feat_idx'].to_numpy()]
    y_train = tr['label'].to_numpy()
    X_test = X[te['feat_idx'].to_numpy()]
    y_test = te['label'].to_numpy()

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    inner_cv = StratifiedKFold(n_splits=INNER_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    gs = GridSearchCV(
        estimator=SVC(),
        param_grid=PARAM_GRID,
        scoring='f1_macro',
        cv=inner_cv,
        n_jobs=GRID_N_JOBS,
        refit=True,
    )

    print(f'Running fold {fold} | train={len(y_train)} test={len(y_test)} | n_jobs={GRID_N_JOBS}')
    with parallel_backend(PARALLEL_BACKEND, n_jobs=GRID_N_JOBS):
        gs.fit(X_train_s, y_train)

    y_pred = gs.predict(X_test_s)
    acc = accuracy_score(y_test, y_pred)
    f1m = f1_score(y_test, y_pred, average='macro')
    cm = confusion_matrix(y_test, y_pred, labels=label_order)
    agg_cm += cm

    all_fold_metrics.append({
        'fold': fold,
        'n_train': int(len(y_train)),
        'n_test': int(len(y_test)),
        'best_C': gs.best_params_['C'],
        'best_gamma': gs.best_params_['gamma'],
        'best_cv_f1_macro': float(gs.best_score_),
        'test_accuracy': float(acc),
        'test_f1_macro': float(f1m),
    })

    rep = classification_report(
        y_test, y_pred, labels=label_order, target_names=class_names_ordered, output_dict=True, zero_division=0
    )
    pd.DataFrame(rep).T.to_csv(STRICT_RESULTS_ROOT / f'{EVAL_SPLIT_SCHEMA}_{fold}_classification_report.csv')

    pred_df = te[['file_name', 'class_name', 'label']].copy()
    pred_df['pred_label'] = y_pred
    pred_df['pred_class'] = pred_df['pred_label'].map(idx_to_class)
    pred_df['fold'] = fold
    all_pred_rows.append(pred_df)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names_ordered, yticklabels=class_names_ordered)
    plt.title(f'Confusion Matrix - {EVAL_SPLIT_SCHEMA} {fold}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(STRICT_FIG_ROOT / f'cm_{EVAL_SPLIT_SCHEMA}_{fold}.png', dpi=220)
    plt.close()

fold_metrics_df = pd.DataFrame(all_fold_metrics)
pred_all_df = pd.concat(all_pred_rows, ignore_index=True)

fold_metrics_df.to_csv(STRICT_RESULTS_ROOT / f'{EVAL_SPLIT_SCHEMA}_fold_metrics.csv', index=False)
pred_all_df.to_csv(STRICT_RESULTS_ROOT / f'{EVAL_SPLIT_SCHEMA}_file_predictions.csv', index=False)
np.save(STRICT_RESULTS_ROOT / f'{EVAL_SPLIT_SCHEMA}_aggregated_confusion_matrix.npy', agg_cm)

plt.figure(figsize=(8, 6))
sns.heatmap(agg_cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names_ordered, yticklabels=class_names_ordered)
plt.title(f'Aggregated Confusion Matrix - {EVAL_SPLIT_SCHEMA}')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(STRICT_FIG_ROOT / f'cm_{EVAL_SPLIT_SCHEMA}_aggregated.png', dpi=220)
plt.close()

summary = {
    'schema': EVAL_SPLIT_SCHEMA,
    'num_folds': int(len(folds)),
    'accuracy_mean': float(fold_metrics_df['test_accuracy'].mean()),
    'accuracy_std': float(fold_metrics_df['test_accuracy'].std(ddof=0)),
    'f1_macro_mean': float(fold_metrics_df['test_f1_macro'].mean()),
    'f1_macro_std': float(fold_metrics_df['test_f1_macro'].std(ddof=0)),
    'strict_replication': True,
}
(STRICT_RESULTS_ROOT / f'{EVAL_SPLIT_SCHEMA}_summary.json').write_text(json.dumps(summary, indent=2))

print(fold_metrics_df)
print('Summary:', json.dumps(summary, indent=2))
print('Saved:', STRICT_RESULTS_ROOT / f'{EVAL_SPLIT_SCHEMA}_summary.json')
